<a href="https://colab.research.google.com/github/amiralito/OpenDDE_Colab/blob/main/OpenDDE_single.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenDDE — single prediction (Colab)

Run **[OpenDDE](https://github.com/aurekaresearch/OpenDDE)** (Aureka Research) end-to-end on
Colab for a **single complex**: protein chains, optional DNA/RNA, ligands (CCD / SMILES /
file), ions, and modified residues. OpenDDE is an **AlphaFold-3-class all-atom co-folding
model** using the AlphaFold-Server input schema.

**Setup is one-time and persistent:** cell 2 installs OpenDDE into an isolated venv and
auto-handles Blackwell (sm_120) GPUs; cell 3b mounts Drive; cell 3c downloads the ~2.6 GB
checkpoint and ~0.6 GB common files **once** — later sessions reuse them from Drive.

**Order:** 1 → 2 → 3 → 3b → 3c → pick ONE input cell → Run settings → 6 → analysis cells.

> Preview software — CLI flags, JSON fields, and checkpoints may change between versions.


In [ ]:
#@title 1. GPU check { display-mode: "form" }
import subprocess
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version,compute_cap',
                          '--format=csv,noheader'], capture_output=True, text=True,
                         check=True).stdout.strip()
    name, mem, drv, cc = [x.strip() for x in out.split(',')]
    gib = int(mem.split()[0]) / 1024
    print(f'GPU: {name} | {gib:.0f} GiB | driver {drv} | compute_cap {cc}')
    if gib < 15:
        print('\n[!] <16 GiB VRAM - fine for short monomers, will OOM on larger complexes.')
    elif gib < 38:
        print('\n[ok] Good for monomers / small complexes. Big assemblies want A100 40-80 GB.')
    else:
        print('\n[ok] Plenty of VRAM for most inputs.')
    if cc.replace('.', '') == '120':
        print('[note] Blackwell (sm_120): cell 2 auto-installs a matching torch and forces the '
              'PyTorch triangle kernels.')
except Exception:
    print('[!] No GPU detected. Runtime -> Change runtime type -> GPU, then rerun.')

In [ ]:
#@title 2. Install OpenDDE + tools (run once per session) { display-mode: "form" }
# OpenDDE pins its own torch, so it lives in an isolated uv venv and never disturbs Colab's.
# No torch is imported in THIS cell on purpose: the Blackwell auto-fix below can therefore
# reinstall torch inside the venv without needing a runtime restart.
import os, sys, subprocess, shutil, time

ENV_DIR     = "/content/opendde_env"
OPENDDE_BIN = f"{ENV_DIR}/bin/opendde"
VPY         = f"{ENV_DIR}/bin/python"
torch_backend = "cu126"  #@param ["cu126", "cu128", "cu124", "cpu"]

def sh(cmd, check=True):
    print('+', cmd); return subprocess.run(cmd, shell=True, check=check)

t0 = time.time()

# 1) uv (only if missing).
if shutil.which('uv') is None and not os.path.exists(os.path.expanduser('~/.local/bin/uv')):
    sh('curl -LsSf https://astral.sh/uv/install.sh | sh')
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
UV = shutil.which('uv') or os.path.expanduser('~/.local/bin/uv')

# 2) OpenDDE (only if missing, so re-running never disturbs a torch fix).
#    opendde is not on PyPI - it installs from the GitHub repo.
GIT_URL = 'https://github.com/aurekaresearch/OpenDDE.git'
opendde_ref = "main"  #@param {type:"string"}
#@markdown `opendde_ref`: `main` tracks the latest release; pin a tag (e.g. `v1.0.3`) for a
#@markdown reproducible run. OpenDDE is not on PyPI, so it installs from the GitHub repo.
_ref = (opendde_ref or 'main').strip()
_spec_ref = '' if _ref in ('', 'main') else f'@{_ref}'
if not os.path.exists(OPENDDE_BIN):
    sh(f'{UV} venv --python 3.11 {ENV_DIR}')
    _extra = 'cpu' if torch_backend == 'cpu' else 'gpu'
    r = subprocess.run([UV, 'pip', 'install', '--python', VPY, '--torch-backend', torch_backend,
                        f'opendde[{_extra}] @ git+{GIT_URL}{_spec_ref}'], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-2500:])
        print('\n[fallback] cloning and installing editable ...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', _ref, GIT_URL,
                        '/content/OpenDDE'], check=False)
        subprocess.run([UV, 'pip', 'install', '--python', VPY, '--torch-backend', torch_backend,
                        '-e', f'.[{_extra}]'], cwd='/content/OpenDDE', check=True)
else:
    print('opendde already installed - skipping its install.')
    print('  (to switch versions: !rm -rf /content/opendde_env, then re-run this cell)')

# Report the installed version so every run is traceable to a release.
_v = subprocess.run([VPY, '-c', 'import opendde; print(opendde.__version__)'],
                    capture_output=True, text=True).stdout.strip()
OPENDDE_VERSION = _v or 'unknown'
print(f'OpenDDE version: {OPENDDE_VERSION}')
if OPENDDE_VERSION not in ('unknown', '') and tuple(
        int(x) for x in OPENDDE_VERSION.split('.')[:3] if x.isdigit()) < (1, 0, 3):
    print('  [!] < 1.0.3: ion entities were dropped from MSA/template metadata, which shifts '
          'chain mappings for ion-containing inputs. Reinstall from main if you use ions.')
for _p in ('gemmi', 'py3Dmol'):
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', _p], check=False)

# 3) Online ColabFold MSA server (OpenDDE hosts no MSA server of its own).
os.environ['MMSEQS_SERVICE_HOST_URL'] = 'https://api.colabfold.com'
# 4) Safe PyTorch LayerNorm path (OpenDDE's default; set explicitly so it survives any env edit).
os.environ['LAYERNORM_TYPE'] = 'torch'

# 5) GPU compute capability via nvidia-smi - NO torch import here, on purpose.
try:
    cc = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                        capture_output=True, text=True).stdout.strip().splitlines()[0].strip()
except Exception:
    cc = ''
os.environ['TORCH_CUDA_ARCH_LIST'] = cc          # '12.0' Blackwell, '8.0' A100, etc.
FORCE_TORCH_KERNELS = False

# 6) Blackwell (sm_120): cu126 wheels lack its kernels. We check the VENV's torch in a
#    SUBPROCESS (nothing imported here) and install the nightly cu128 build only if needed.
#    cuEquivariance also ships no sm_120 triangle kernels yet, so we fall back to the
#    documented PyTorch kernel path (--trimul_kernel/--triatt_kernel torch).
if cc.replace('.', '') == '120':
    _chk = subprocess.run([VPY, '-c',
        "import torch,sys; sys.exit(0 if 'sm_120' in ' '.join(torch.cuda.get_arch_list()) else 1)"],
        capture_output=True)
    if _chk.returncode == 0:
        print('** Blackwell (sm_120): venv torch already carries sm_120 kernels - good to go. **')
    else:
        print('** Blackwell (sm_120) detected: installing nightly cu128 torch into the venv '
              'automatically (~2-3 min, one-time). **')
        subprocess.run([UV, 'pip', 'install', '--python', VPY, '--pre', '--reinstall', 'torch',
                        '--index-url', 'https://download.pytorch.org/whl/nightly/cu128'], check=False)
        print('   Done - no restart needed. Continue to the next cells.')
    FORCE_TORCH_KERNELS = True
    print('   Triangle kernels forced to the PyTorch path for sm_120 (see docs/kernels.md).')

def opendde(args, capture=False, quiet=False):
    """Run an opendde subcommand against the venv binary + current model root."""
    env = os.environ.copy()
    env['OPENDDE_ROOT_DIR'] = globals().get('OPENDDE_ROOT_DIR', os.path.expanduser('~/.cache/opendde'))
    cmd = [OPENDDE_BIN] + [str(a) for a in args]
    if not quiet: print('$', ' '.join(cmd))
    return (subprocess.run(cmd, env=env, capture_output=True, text=True) if capture
            else subprocess.run(cmd, env=env))

print(f'\nSetup done in {time.time()-t0:.0f}s | compute_cap {cc or "n/a"} | LAYERNORM_TYPE=torch')
print('MMSEQS_SERVICE_HOST_URL =', os.environ['MMSEQS_SERVICE_HOST_URL'])
print('Note: torch is not imported in this cell, so the Blackwell auto-fix needs no restart.')

In [ ]:
#@title 2c. Blackwell (sm_120) torch — optional fallback (only if cell 2 auto-fix fails) { display-mode: "form" }
# Last resort: force the nightly cu128 torch INTO THE OPENDDE VENV (not Colab's python).
import os, subprocess, shutil
os.environ['TORCH_CUDA_ARCH_LIST'] = '12.0'
UV  = shutil.which('uv') or os.path.expanduser('~/.local/bin/uv')
VPY = '/content/opendde_env/bin/python'
subprocess.run([UV, 'pip', 'install', '--python', VPY, '--pre', '--reinstall', 'torch',
                '--index-url', 'https://download.pytorch.org/whl/nightly/cu128'], check=True)
print('\nDONE. Re-run cell 2 (no restart needed - the venv torch is separate from Colab\'s).')
print('Verify:  !/content/opendde_env/bin/python -c "import torch;print(torch.cuda.get_arch_list())"')
print('  -> must include sm_120')

In [ ]:
#@title 3. Helpers (run once) { display-mode: "form" }
import os, json, glob, time

custom_output_dir = ""  #@param {type:"string"}

WORK      = '/content/opendde_run'
INPUT_DIR = os.path.join(WORK, 'inputs')
MSA_DIR   = os.path.join(WORK, 'msa')
OUT_DIR   = custom_output_dir.strip() or os.path.join(WORK, 'outputs')
# OpenDDE reads its checkpoint + common files from OPENDDE_ROOT_DIR (redirected to Drive in 3b).
OPENDDE_ROOT_DIR = os.path.join(WORK, 'opendde_data')
for d in (INPUT_DIR, MSA_DIR, OUT_DIR, OPENDDE_ROOT_DIR):
    os.makedirs(d, exist_ok=True)
print('Output dir:', OUT_DIR)

JOB_SPEC = None  # set by an input cell

def _clean_seq(s):
    return ''.join(s.split()).upper()

def protein(seq, count=1, ids=None):
    d = {'sequence': _clean_seq(seq), 'count': int(count)}
    if ids: d['id'] = ids
    return {'proteinChain': d}

def ligand(spec, count=1):
    # spec: 'CCD_ATP'  OR  a SMILES string  OR  'FILE_/path/to/lig.sdf'
    return {'ligand': {'ligand': spec, 'count': int(count)}}

def ion(code, count=1):
    # code is a bare CCD code WITHOUT the CCD_ prefix, e.g. 'MG', 'ZN', 'NA'
    return {'ion': {'ion': code, 'count': int(count)}}

def dna(seq, count=1):
    return {'dnaSequence': {'sequence': _clean_seq(seq), 'count': int(count)}}

def rna(seq, count=1):
    return {'rnaSequence': {'sequence': _clean_seq(seq), 'count': int(count)}}

def make_spec(name, entities):
    name = name.strip().replace(' ', '_') or 'job'
    return {'name': name, 'sequences': entities}

def write_json(spec, path=None):
    path = path or os.path.join(INPUT_DIR, spec['name'] + '.json')
    with open(path, 'w') as f:
        json.dump([spec], f, indent=2)   # top level must be a LIST
    print('Wrote', path)
    print(json.dumps([spec], indent=2)[:1200])
    return path

# --------------------------------------------------------------------------- #
# weights + common files
# --------------------------------------------------------------------------- #
HF_REPO   = 'aurekaresearch/OpenDDE'
HF_BASE   = f'https://huggingface.co/{HF_REPO}/resolve/main'
CKPTS     = {'opendde_v1': 'opendde.pt', 'abag': 'opendde_abag.pt'}   # ~2.6 GB each
# Runtime files OpenDDE otherwise fetches on first inference (~0.6 GB total).
COMMON    = ['components.cif', 'components.cif.rdkit_mol.pkl',
             'obsolete_to_successor.json', 'release_date_cache.json']

def _fast_download(repo_file, out, min_bytes=1_000_000):
    """Download `repo_file` from the OpenDDE HF repo to `out` as fast as the runtime allows:
       hf_transfer (parallel, xet-aware), then aria2c (16 connections), then curl."""
    import subprocess, importlib.util
    os.makedirs(os.path.dirname(out), exist_ok=True)
    # 1) huggingface_hub + hf_transfer - best for HF (xet-backed) repos
    try:
        if importlib.util.find_spec('hf_transfer') is None:
            subprocess.run('pip -q install hf_transfer huggingface_hub', shell=True, check=False)
        os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=HF_REPO, filename=repo_file, local_dir='/content/_hf_stage')
        if os.path.abspath(p) != os.path.abspath(out):
            import shutil; shutil.move(p, out)
        if os.path.exists(out) and os.path.getsize(out) > min_bytes:
            return True
    except Exception as e:
        print('  (hf_transfer path unavailable:', str(e)[:100], '- falling back)')
    url = f'{HF_BASE}/{repo_file}'
    # 2) aria2c - many parallel connections
    if subprocess.run('which aria2c', shell=True, capture_output=True).returncode != 0:
        subprocess.run('apt-get -qq install -y aria2 >/dev/null 2>&1 || true', shell=True, check=False)
    if subprocess.run('which aria2c', shell=True, capture_output=True).returncode == 0:
        d, f = os.path.dirname(out), os.path.basename(out)
        rc = subprocess.run(f'aria2c -x16 -s16 -k1M -c --console-log-level=warn -d "{d}" -o "{f}" "{url}"',
                            shell=True).returncode
        if rc == 0 and os.path.exists(out) and os.path.getsize(out) > min_bytes:
            return True
    # 3) curl fallback
    subprocess.run(f'curl -fL --retry 4 --retry-delay 3 -o "{out}" "{url}"', shell=True, check=False)
    return os.path.exists(out) and os.path.getsize(out) > min_bytes

def ensure_opendde_weights(model='opendde_v1', dest_dir=None, with_common=True, verbose=True):
    """Stage the OpenDDE checkpoint (+ common runtime files) under OPENDDE_ROOT_DIR, which
       points at Drive after cell 3b - so this downloads ONCE and later sessions reuse it.
       Returns the checkpoint path. Safe to call every run: present files are skipped."""
    root = (dest_dir.strip() if isinstance(dest_dir, str) and dest_dir.strip()
            else globals().get('OPENDDE_ROOT_DIR', os.path.expanduser('~/.cache/opendde')))
    fname = CKPTS.get(model, CKPTS['opendde_v1'])
    ckpt  = os.path.join(root, 'checkpoint', fname)
    if os.path.exists(ckpt) and os.path.getsize(ckpt) > 1e9:
        if verbose: print(f'Found checkpoint: {ckpt} ({os.path.getsize(ckpt)/1e9:.2f} GB) - no download.')
    else:
        if verbose: print(f'Downloading {fname} (~2.6 GB, one-time) -> {ckpt} ...')
        if _fast_download(fname, ckpt, min_bytes=1e9):
            if verbose: print(f'  -> saved ({os.path.getsize(ckpt)/1e9:.2f} GB). Future runs reuse this file.')
        else:
            print('  -> download failed; check connectivity and retry.')
    if with_common:
        cdir = os.path.join(root, 'common'); os.makedirs(cdir, exist_ok=True)
        for cf in COMMON:
            dst = os.path.join(cdir, cf)
            if os.path.exists(dst) and os.path.getsize(dst) > 1000:
                continue
            if verbose: print(f'  fetching common/{cf} ...')
            _fast_download(f'common/{cf}', dst, min_bytes=1000)
    os.environ['OPENDDE_ROOT_DIR'] = root
    return ckpt

print('Helpers ready. Mount Drive (3b) next so weights and MSAs persist.')

In [ ]:
#@title 3b. (Recommended) Save outputs to Google Drive so a disconnect can't lose them { display-mode: "form" }
# Colab wipes /content the moment the runtime disconnects. Mount Drive and redirect the work
# dirs so predictions, the MSA cache AND the ~3 GB model files persist across sessions.
# Run this AFTER cell 3 and BEFORE the weights cell.
from google.colab import drive
import os, shutil, time
_mp = '/content/drive'
if os.path.ismount(_mp):
    print('Drive already mounted - skipping mount.')
else:
    if os.path.isdir(_mp) and os.listdir(_mp):
        # not a live mount but has leftover files that block mounting -> move aside (preserve)
        _bak = f'{_mp}_old_{int(time.time())}'
        shutil.move(_mp, _bak)
        print(f'Moved stale {_mp} -> {_bak} (leftover files, not live Drive).')
    drive.mount(_mp)

WORK             = '/content/drive/MyDrive/opendde_run'
INPUT_DIR        = os.path.join(WORK, 'inputs')
MSA_DIR          = os.path.join(WORK, 'msa')
OPENDDE_ROOT_DIR = os.path.join(WORK, 'opendde_data')
_custom          = custom_output_dir.strip() if 'custom_output_dir' in globals() else ''
OUT_DIR          = _custom or os.path.join(WORK, 'outputs')
for d in (INPUT_DIR, MSA_DIR, OUT_DIR, OPENDDE_ROOT_DIR):
    os.makedirs(d, exist_ok=True)
os.environ['OPENDDE_ROOT_DIR'] = OPENDDE_ROOT_DIR
print('Outputs will persist to:', OUT_DIR)
print('Model files (checkpoint + common) will persist to:', OPENDDE_ROOT_DIR)

In [ ]:
#@title 3c. OpenDDE weights (downloads once, reused after) { display-mode: "form" }
model_weights = "opendde_v1 (general)"  #@param ["opendde_v1 (general)", "abag (antibody-antigen)"]
weights_dir = ""  #@param {type:"string"}
prefetch_common_files = True  #@param {type:"boolean"}
# Blank weights_dir = default (OPENDDE_ROOT_DIR, on Drive after 3b).
# Prefetching the common files (~0.6 GB) avoids a mid-run download on first inference.
_dest  = weights_dir.strip() or None
_model = 'abag' if model_weights.startswith('abag') else 'opendde_v1'
CKPT_PATH = ensure_opendde_weights(model=_model, dest_dir=_dest,
                                   with_common=prefetch_common_files)
print('Checkpoint:', CKPT_PATH)

In [ ]:
#@title Monomer { display-mode: "form" }
name = "my_monomer"  #@param {type:"string"}
sequence = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"  #@param {type:"string"}
JOB_SPEC = make_spec(name, [protein(sequence, 1)])
JOB_JSON = write_json(JOB_SPEC)

In [ ]:
#@title Multimer { display-mode: "form" }
#@markdown Leave a sequence blank to skip that chain. `count` = copies of that chain.
name = "my_complex"  #@param {type:"string"}
chain1_seq = ""  #@param {type:"string"}
chain1_count = 1  #@param {type:"integer"}
chain2_seq = ""  #@param {type:"string"}
chain2_count = 1  #@param {type:"integer"}
chain3_seq = ""  #@param {type:"string"}
chain3_count = 1  #@param {type:"integer"}
chain4_seq = ""  #@param {type:"string"}
chain4_count = 1  #@param {type:"integer"}
_ents = [protein(s, c) for s, c in
         [(chain1_seq, chain1_count), (chain2_seq, chain2_count),
          (chain3_seq, chain3_count), (chain4_seq, chain4_count)] if s.strip()]
assert _ents, 'Enter at least one sequence.'
JOB_SPEC = make_spec(name, _ents)
JOB_JSON = write_json(JOB_SPEC)

In [ ]:
#@title Homomer { display-mode: "form" }
name = "my_homomer"  #@param {type:"string"}
sequence = ""  #@param {type:"string"}
n_copies = 3  #@param {type:"integer"}
assert sequence.strip(), 'Enter a sequence.'
JOB_SPEC = make_spec(name, [protein(sequence, int(n_copies))])
JOB_JSON = write_json(JOB_SPEC)

In [ ]:
#@title Protein + ligand { display-mode: "form" }
#@markdown `ligand` accepts a CCD code (`CCD_ATP`), underscore-joined CCD codes
#@markdown (`CCD_NAG_BMA_BGC`), a SMILES string, or a file (`FILE_/path/lig.sdf`).
#@markdown `ion` codes are bare CCD names (e.g. `MG`, `ZN`).
name = "my_complex_lig"  #@param {type:"string"}
protein_seq = ""  #@param {type:"string"}
protein_count = 1  #@param {type:"integer"}
ligand_spec = "CCD_ATP"  #@param {type:"string"}
ligand_count = 1  #@param {type:"integer"}
ion_code = ""  #@param {type:"string"}
ion_count = 1  #@param {type:"integer"}
assert protein_seq.strip(), 'Enter a protein sequence.'
_ents = [protein(protein_seq, int(protein_count))]
if ligand_spec.strip(): _ents.append(ligand(ligand_spec.strip(), int(ligand_count)))
if ion_code.strip():    _ents.append(ion(ion_code.strip(), int(ion_count)))
JOB_SPEC = make_spec(name, _ents)
JOB_JSON = write_json(JOB_SPEC)

In [ ]:
#@title Run settings { display-mode: "form" }
model_name = "opendde_v1"  #@param ["opendde_v1"]
checkpoint = "general (opendde.pt)"  #@param ["general (opendde.pt)", "antibody-antigen (opendde_abag.pt)"]
seeds_mode = "list"  #@param ["list", "random_n"]
seeds_list = "101,102,103,104,105"  #@param {type:"string"}
random_n = 5  #@param {type:"slider", min:1, max:25, step:1}
samples_per_seed = 5  #@param {type:"slider", min:1, max:25, step:1}
steps = 200  #@param {type:"integer"}
cycles = 10  #@param {type:"integer"}
dtype = "fp32"  #@param ["fp32", "bf16"]
use_msa = True  #@param {type:"boolean"}
use_tfg_guidance = False  #@param {type:"boolean"}
need_atom_confidence = True  #@param {type:"boolean"}
triangle_kernels = "auto"  #@param ["auto", "torch", "cuequivariance"]
#@markdown `need_atom_confidence` writes the full-data JSON that the PAE cells read.
#@markdown `triangle_kernels`: `auto` picks cuEquivariance when usable; it is forced to
#@markdown `torch` automatically on Blackwell (sm_120), which has no cuEquivariance kernels yet.
import random, os
if seeds_mode == "list":
    SEEDS = [int(x) for x in seeds_list.split(',') if x.strip()]
else:
    SEEDS = random.sample(range(1, 100000), int(random_n))

_kern = triangle_kernels
if globals().get('FORCE_TORCH_KERNELS') and _kern == 'auto':
    _kern = 'torch'
    print('Blackwell (sm_120): triangle kernels forced to torch.')

CFG = dict(model_name=model_name,
           model_key=('abag' if checkpoint.startswith('antibody') else 'opendde_v1'),
           seeds_csv=','.join(map(str, SEEDS)), n_sample=int(samples_per_seed),
           steps=int(steps), cycles=int(cycles), dtype=dtype, use_msa=use_msa,
           use_tfg=use_tfg_guidance, need_atom_conf=need_atom_confidence, kernels=_kern)
print('Seeds:', SEEDS)
print(CFG)

In [ ]:
#@title 6. MSA + inference { display-mode: "form" }
import os, glob, json, subprocess, datetime, time
assert 'CFG' in globals(),      'Run the Run settings cell first.'
assert 'JOB_SPEC' in globals() and JOB_SPEC, 'Run ONE input cell first.'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ.setdefault('MMSEQS_SERVICE_HOST_URL', 'https://api.colabfold.com')
FORCE_REMSA = False  #@param {type:"boolean"}
VERBOSE = False      #@param {type:"boolean"}

# Weights staged automatically (skipped if already on Drive from a previous session).
CKPT_PATH = ensure_opendde_weights(model=CFG['model_key'], verbose=True)

def _run(cmd):
    print('\n+ ' + ' '.join(str(c) for c in cmd), flush=True)
    e = dict(os.environ); e['OPENDDE_ROOT_DIR'] = OPENDDE_ROOT_DIR
    p = subprocess.Popen([str(c) for c in cmd], env=e, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    buf = []
    for line in p.stdout:
        buf.append(line); low = line.lower()
        keep = (VERBOSE
                or any(k in low for k in ('error', 'traceback', 'failed', 'exception',
                                          'out of memory', 'warning', 'file "'))
                or any(k in line for k in ('completed', 'Downloading', 'Saved', 'Seed',
                                           'Predict', 'Inference', 'N_token', 'Resolved'))
                or '%|' in line or 'it/s' in line or 's/it' in line)
        if keep: print(line, end='')
    p.wait(); return p.returncode, ''.join(buf)

JOB = JOB_SPEC['name']
predict_json = JOB_JSON
if CFG['use_msa']:
    cached = os.path.splitext(JOB_JSON)[0] + '-update-msa.json'
    if os.path.exists(cached) and not FORCE_REMSA:
        predict_json = cached; print('Reusing cached MSA JSON:', cached)
    else:
        j_msa = os.path.join(MSA_DIR, JOB); os.makedirs(j_msa, exist_ok=True)
        rc, _ = _run([OPENDDE_BIN, 'msa', '-i', JOB_JSON, '-o', j_msa])
        if rc != 0:
            print('[!] MSA step failed - falling back to single-sequence inference.')
            CFG['use_msa'] = False
        elif os.path.exists(cached):
            predict_json = cached; print('Using MSA-annotated JSON:', cached)

_stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
RUN_DIR = os.path.join(OUT_DIR, f'run_{_stamp}_{JOB}')
os.makedirs(RUN_DIR, exist_ok=True)
print('This run ->', RUN_DIR)

cmd = [OPENDDE_BIN, 'pred', '-i', predict_json, '-o', RUN_DIR, '-n', CFG['model_name'],
       '--seeds', CFG['seeds_csv'], '--sample', CFG['n_sample'], '--step', CFG['steps'],
       '--cycle', CFG['cycles'], '--dtype', CFG['dtype'],
       '--use_msa', str(CFG['use_msa']).lower(),
       '--use_template', 'false', '--use_rna_msa', 'false',
       '--load_checkpoint_path', CKPT_PATH]
if CFG['kernels'] != 'auto':
    cmd += ['--trimul_kernel', CFG['kernels'], '--triatt_kernel', CFG['kernels']]
if CFG['need_atom_conf']: cmd += ['--need_atom_confidence', 'true']
if CFG['use_tfg']:        cmd += ['--use_tfg_guidance', 'true']

t0 = time.time()
rc, out = _run(cmd)
n_cif = len(glob.glob(os.path.join(RUN_DIR, '**', '*.cif'), recursive=True))
print(f'\nExit {rc} in {time.time()-t0:.0f}s | CIFs: {n_cif}')
assert rc == 0 and n_cif > 0, 'Prediction failed - full log printed above.'

In [ ]:
#@title 7. Rank predictions by confidence { display-mode: "form" }
import glob, os, json
import pandas as pd, numpy as np

assert JOB_SPEC is not None, (
    "JOB_SPEC not set. For a BATCH: run B3 (it auto-selects the top hit) or set "
    "JOB_SPEC = {'name': '<job>'}. For a SINGLE run: run an input cell 4a-4d.")
_RUN = globals().get('RUN_DIR', OUT_DIR)
job_out = os.path.join(_RUN, JOB_SPEC['name'])
rows = []
for jf in glob.glob(os.path.join(job_out, '**', '*summary_confidence*.json'), recursive=True):
    try:
        d = json.load(open(jf))
    except Exception:
        continue
    cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
    # interface confidence = weakest off-diagonal chain-pair ipTM (None for monomers)
    cpi = np.array((d.get('chain_pair_iptm_global') or (d.get('chain_pair_iptm_global') or d.get('chain_pair_iptm')) or []), dtype=float)
    if cpi.ndim == 2 and cpi.shape[0] > 1:
        iface = float(cpi[~np.eye(cpi.shape[0], dtype=bool)].min())
    else:
        iface = None
    seed = jf.split(os.sep)[-3] if os.sep + 'seed' in jf else ''
    rows.append({
        'seed': seed,
        'cif': cif if os.path.exists(cif) else '(missing)',
        'summary': jf,
        'ranking_score': d.get('ranking_score'),
        'iface_iptm': iface,
        'iptm': d.get('iptm'),
        'ptm': d.get('ptm'),
        'plddt': d.get('plddt'),
        'gpde': d.get('gpde'),
        'disorder': d.get('disorder'),
        'has_clash': d.get('has_clash'),
    })

assert rows, 'No summary_confidence JSONs found. Did cell 6 finish?'
df = pd.DataFrame(rows).sort_values('ranking_score', ascending=False, na_position='last').reset_index(drop=True)
pd.set_option('display.max_colwidth', 80)
display(df.drop(columns=['summary']))
BEST_CIF = df.iloc[0]['cif']
BEST_SUMMARY = df.iloc[0]['summary']
print('\nBest model:', BEST_CIF)
if df.iloc[0]['iface_iptm'] is not None:
    print(f"Interface ipTM of best model: {df.iloc[0]['iface_iptm']:.3f}  "
          "(>0.8 confident, 0.6-0.8 plausible, <0.6 weak)")

In [ ]:
#@title Model viewer (MolView) + scores { display-mode: "form" }
# Pick any predicted model from the dropdowns; it renders in MolView (Mol*-based) colored by
# pLDDT, with its confidence scores shown. Works for a single job or a whole batch.
import os, glob, json, re, subprocess, sys
import numpy as np, pandas as pd

try:
    import molview as mv
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'molview'], check=False)
    import molview as mv
import ipywidgets as widgets
from IPython.display import display, clear_output
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

_RUN = globals().get('RUN_DIR', OUT_DIR)

def _seed_of(p):
    for x in p.split(os.sep):
        if x.startswith('seed_'):
            return x.split('seed_')[-1]
    for x in p.split(os.sep):
        if x.isdigit():
            return x
    return 'NA'

def _interface_iptm(d):
    try:
        a = np.array((d.get('chain_pair_iptm_global') or d.get('chain_pair_iptm')), dtype=float)
        if a.ndim == 2 and a.shape[0] > 1:
            off = a.copy(); np.fill_diagonal(off, np.nan)
            return float(np.nanmax(off))
    except Exception:
        pass
    return float('nan')

def _cif_for(jf):
    # same derivation cell 7 uses, with a glob fallback
    cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
    if os.path.exists(cif):
        return cif
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    d = os.path.dirname(jf)
    cand = (glob.glob(os.path.join(d, f'*sample_{ms.group(1)}.cif')) if ms
            else glob.glob(os.path.join(d, '*.cif')))
    return cand[0] if cand else None

# optional ipSAE table (present only after the ipSAE cell has run)
_ipsae = None
_ip_csv = os.path.join(_RUN, 'ipsae_all_chain_pairs.csv')
if os.path.exists(_ip_csv):
    try:
        _ipsae = pd.read_csv(_ip_csv)
        _ipsae['seed'] = _ipsae['seed'].astype(str)
        _ipsae['ipSAE'] = pd.to_numeric(_ipsae.get('ipSAE'), errors='coerce')
    except Exception:
        _ipsae = None

# discover models the same way cell 7 / B-cells do: via the summary JSONs
recs = []
for jf in glob.glob(os.path.join(_RUN, '**', '*summary_confidence*.json'), recursive=True):
    try:
        d = json.load(open(jf))
    except Exception:
        continue
    cif = _cif_for(jf)
    if not cif:
        continue
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    recs.append(dict(job=os.path.relpath(jf, _RUN).split(os.sep)[0],
                     seed=_seed_of(jf), sample=int(ms.group(1)) if ms else 0,
                     cif=cif, ranking=d.get('ranking_score', float('nan')),
                     iptm=d.get('iptm', float('nan')), ptm=d.get('ptm', float('nan')),
                     iiptm=_interface_iptm(d), plddt=d.get('plddt'),
                     has_clash=d.get('has_clash')))
mdf = pd.DataFrame(recs)

if mdf.empty:
    print('No models found under', _RUN)
    print('Looked for *summary_confidence*.json - has the inference cell finished and written here?')
else:
    mdf = mdf.sort_values('ranking', ascending=False).reset_index(drop=True)
    jobs = list(dict.fromkeys(mdf['job']))            # best-ranked job first
    job_dd   = widgets.Dropdown(options=jobs, description='Job:',
                                layout=widgets.Layout(width='70%'))
    model_dd = widgets.Dropdown(description='Model:', layout=widgets.Layout(width='70%'))
    color_dd = widgets.Dropdown(options=['plddt', 'chain', 'rainbow', 'secondary'],
                                value='plddt', description='Color:',
                                layout=widgets.Layout(width='40%'))
    out = widgets.Output()

    def _opts(job):
        sub = mdf[mdf['job'] == job]
        return [(f"seed {r['seed']} \u00b7 sample {r['sample']}  |  ipTM {r['iptm']:.2f}  "
                 f"pTM {r['ptm']:.2f}  rank {r['ranking']:.3f}", r['cif'])
                for _, r in sub.iterrows()]

    def _render(cif):
        with out:
            clear_output(wait=True)
            r = mdf[mdf['cif'] == cif].iloc[0]
            row = {'plddt': r['plddt'], 'pTM': r['ptm'], 'ipTM': r['iptm'],
                   'interface_ipTM': r['iiptm'], 'ranking_score': r['ranking'],
                   'has_clash': r['has_clash']}
            if _ipsae is not None:
                sel = _ipsae[(_ipsae['job'] == r['job']) & (_ipsae['seed'] == str(r['seed']))]
                if len(sel) and 'ipSAE' in sel.columns:
                    row['ipSAE (max pair)'] = float(sel['ipSAE'].max())
            print(f"{r['job']}  |  seed {r['seed']}  \u00b7  sample {r['sample']}")
            display(pd.DataFrame([row]).T.rename(columns={0: 'value'}))
            try:
                v = mv.view(width=720, height=520, panel=True)
                with open(cif) as f:
                    v.addModel(f.read())
                v.setColorMode(color_dd.value)
                v.show()
            except Exception as e:
                print('MolView render failed:', e)
                print('Open this file in ChimeraX instead:', cif)

    def _on_job(ch):
        if ch['name'] == 'value':
            model_dd.options = _opts(ch['new'])
            if model_dd.options:
                model_dd.value = model_dd.options[0][1]

    def _on_model(ch):
        if ch['name'] == 'value' and ch['new']:
            _render(ch['new'])

    def _on_color(ch):
        if ch['name'] == 'value' and model_dd.value:
            _render(model_dd.value)

    job_dd.observe(_on_job, names='value')
    model_dd.observe(_on_model, names='value')
    color_dd.observe(_on_color, names='value')

    display(widgets.HBox([job_dd]), widgets.HBox([model_dd, color_dd]), out)
    model_dd.options = _opts(jobs[0])
    if model_dd.options:
        model_dd.value = model_dd.options[0][1]
        _render(model_dd.value)

In [ ]:
#@title 9. Per-residue pLDDT + interface confidence { display-mode: "form" }
import gemmi, numpy as np, json, os
import matplotlib.pyplot as plt

assert 'BEST_CIF' in globals() and 'BEST_SUMMARY' in globals(), \
    'Run cell 7 first - it sets BEST_CIF / BEST_SUMMARY (after a batch, run B3 then 7).'

# ---- left: per-residue pLDDT from CIF B-factors ----
st = gemmi.read_structure(BEST_CIF)
model = st[0]
res_plddt, boundaries, chain_mid, chain_names, idx = [], [], [], [], 0
for chain in model:
    start = idx
    for res in chain:
        bs = [a.b_iso for a in res]
        if bs:
            res_plddt.append(sum(bs)/len(bs)); idx += 1
    if idx > start:
        boundaries.append(idx); chain_mid.append((start+idx)//2); chain_names.append(chain.name)
res_plddt = np.array(res_plddt)

# ---- right: interface confidence (chain_pair_iptm) from the summary JSON ----
d = json.load(open(BEST_SUMMARY))
cpi = np.array((d.get('chain_pair_iptm_global') or (d.get('chain_pair_iptm_global') or d.get('chain_pair_iptm')) or []), dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(range(1, len(res_plddt)+1), res_plddt, lw=1.1, color='#2b6cb0')
ax.axhspan(90,100,alpha=.07,color='blue');  ax.axhspan(70,90,alpha=.07,color='cyan')
ax.axhspan(50,70,alpha=.07,color='orange'); ax.axhspan(0,50,alpha=.07,color='red')
for b in boundaries[:-1]:
    ax.axvline(b+0.5, color='k', ls='--', lw=.6, alpha=.5)
for m, nm in zip(chain_mid, chain_names):
    ax.text(m, 103, nm, ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Residue'); ax.set_ylabel('pLDDT'); ax.set_ylim(0, 108)
ax.set_title(f'Per-residue pLDDT (mean {res_plddt.mean():.1f})')

ax2 = axes[1]
if cpi.ndim == 2 and cpi.size:
    im = ax2.imshow(cpi, cmap='viridis', vmin=0, vmax=1)
    for i in range(cpi.shape[0]):
        for j in range(cpi.shape[1]):
            ax2.text(j, i, f'{cpi[i,j]:.2f}', ha='center', va='center', fontsize=10,
                     color='white' if cpi[i,j] < 0.6 else 'black')
    labs = chain_names if len(chain_names) == cpi.shape[0] else [str(i) for i in range(cpi.shape[0])]
    ax2.set_xticks(range(len(labs))); ax2.set_xticklabels(labs)
    ax2.set_yticks(range(len(labs))); ax2.set_yticklabels(labs)
    fig.colorbar(im, ax=ax2, label='chain-pair ipTM')
    ax2.set_title('Interface confidence (chain_pair_iptm)')
else:
    ax2.axis('off'); ax2.set_title('chain_pair_iptm not available (monomer?)')
plt.tight_layout(); plt.show()

if cpi.ndim == 2 and cpi.shape[0] > 1:
    off = cpi[~np.eye(cpi.shape[0], dtype=bool)]
    print(f'Inter-chain ipTM: min {off.min():.3f}, max {off.max():.3f}  '
          '(>0.8 confident interface, 0.6-0.8 plausible, <0.6 weak/unreliable)')
    print('Tip: agreement of the top interface across seeds matters more than any single value.')

In [ ]:
#@title 9b. Interface contacts & residue-residue reliability (+ per-job table) { display-mode: "form" }
# Builds the inter-chain contact map for each job's best model, lists the closest residue pairs
# with their pLDDT, and reports how often each contact recurs across that job's samples/seeds.
# Exports one table per job (and a combined CSV); shows the map + table for the current top job.
import gemmi, numpy as np, glob, os, json, re
import pandas as pd
import matplotlib.pyplot as plt

_RUN = globals().get('RUN_DIR', OUT_DIR)
CONTACT = 8.0          #@param {type:"number"}
max_rows_print = 30    #@param {type:"integer"}

def load_chains(cif):
    st = gemmi.read_structure(cif); m = st[0]
    out = {}
    for ch in m:
        rl = []
        for res in ch:
            coords = np.array([[a.pos.x, a.pos.y, a.pos.z] for a in res], dtype=float)
            if len(coords) == 0:
                continue
            ca = [[a.pos.x, a.pos.y, a.pos.z] for a in res if a.name == 'CA']
            rep = np.array(ca[0]) if ca else coords.mean(0)
            tab = gemmi.find_tabulated_residue(res.name)
            code = tab.one_letter_code.upper() if tab else 'X'
            rl.append({'num': res.seqid.num, 'code': code, 'rep': rep,
                       'plddt': float(np.mean([a.b_iso for a in res]))})
        if rl:
            out[ch.name] = rl
    return out

def _best_cif(job):
    best, br = None, -1e9
    for jf in glob.glob(os.path.join(_RUN, job, '**', '*summary_confidence*sample_*.json'), recursive=True):
        try:
            r = float(json.load(open(jf)).get('ranking_score') or 0)
        except Exception:
            r = 0.0
        if r > br:
            cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
            if not os.path.exists(cif):
                ms = re.search(r'sample_(\d+)', os.path.basename(jf)); d = os.path.dirname(jf)
                c = (glob.glob(os.path.join(d, f'*sample_{ms.group(1)}.cif')) if ms
                     else glob.glob(os.path.join(d, '*.cif')))
                cif = c[0] if c else None
            if cif:
                br, best = r, cif
    return best

def contacts_table(job, cutoff=CONTACT):
    """Unique inter-chain residue pairs < cutoff for the job's best model, with recurrence
       across the job's samples. Returns (rows, D, A, B, cA, cB, n_used)."""
    cif = _best_cif(job)
    if not cif:
        return [], None, None, None, None, None, 0
    chains = load_chains(cif); names = list(chains)
    if len(names) < 2:
        return [], None, None, None, None, None, 0
    cA, cB = names[0], names[1]                     # first two chains; edit for another pair
    A, B = chains[cA], chains[cB]
    PA = np.array([r['rep'] for r in A]); PB = np.array([r['rep'] for r in B])
    D = np.linalg.norm(PA[:, None, :] - PB[None, :, :], axis=-1)
    freq = np.zeros_like(D); n_used = 0
    for cf in glob.glob(os.path.join(_RUN, job, '**', '*sample_*.cif'), recursive=True):
        ch = load_chains(cf)
        if cA in ch and cB in ch and len(ch[cA]) == len(A) and len(ch[cB]) == len(B):
            pa = np.array([r['rep'] for r in ch[cA]]); pb = np.array([r['rep'] for r in ch[cB]])
            freq += (np.linalg.norm(pa[:, None, :] - pb[None, :, :], axis=-1) < cutoff); n_used += 1
    freq /= max(n_used, 1)
    ii, jj = np.where(D < cutoff)
    rows, seen = [], set()
    for i, j in sorted(zip(ii, jj), key=lambda p: D[p]):
        key = (A[i]['num'], B[j]['num'])
        if key in seen:
            continue
        seen.add(key)
        rows.append(dict(job=job, chainA=cA, resA=A[i]['num'], codeA=A[i]['code'],
                         plddtA=round(A[i]['plddt'], 1), chainB=cB, resB=B[j]['num'],
                         codeB=B[j]['code'], plddtB=round(B[j]['plddt'], 1),
                         distance=round(float(D[i, j]), 2), recurrence=round(float(freq[i, j]), 3)))
    return rows, D, A, B, cA, cB, n_used

# ---- per-job export across every job in the run ----
jobs = sorted(d for d in os.listdir(_RUN)
              if os.path.isdir(os.path.join(_RUN, d)) and not d.startswith('_'))
all_rows = []
for job in jobs:
    rows = contacts_table(job)[0]
    if rows:
        all_rows += rows
        pd.DataFrame(rows).to_csv(os.path.join(_RUN, job, 'interface_contacts.csv'), index=False)

if all_rows:
    df = pd.DataFrame(all_rows)
    out_csv = os.path.join(_RUN, 'interface_contacts_all_jobs.csv')
    df.to_csv(out_csv, index=False)
    print(f'Wrote {out_csv}  |  {len(df)} contacts across {df["job"].nunique()} job(s).')
    print('Per-job copies: <job>/interface_contacts.csv')
else:
    df = None
    print('No inter-chain contacts found (monomers, or no >=2-chain jobs).')

# ---- interactive map + table for the current top job ----
_js = globals().get('JOB_SPEC') or {}
view_job = (_js.get('name') if _js.get('name') in jobs
            else (df.iloc[0]['job'] if df is not None else (jobs[0] if jobs else None)))
if view_job:
    rows, D, A, B, cA, cB, n_used = contacts_table(view_job)
    if D is not None:
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(D, cmap='viridis_r', vmin=0, vmax=20, aspect='auto')
        ax.set_xlabel(f'chain {cB} residue'); ax.set_ylabel(f'chain {cA} residue')
        ax.set_title(f'{view_job}: inter-chain distances | contacts < {CONTACT} A '
                     f'(recurrence over {n_used} models)')
        fig.colorbar(im, ax=ax, label='distance (Angstrom)'); plt.tight_layout(); plt.show()
        from IPython.display import display
        print(f'\nTop {min(max_rows_print, len(rows))} of {len(rows)} contacts for {view_job} '
              f'(chains {cA}-{cB}):\n')
        display(pd.DataFrame(rows).head(max_rows_print))
        print('\nHigh recurrence (~1.0) + high pLDDT on both partners = trustworthy specific contact.')

In [ ]:
#@title B5. PAE - best per seed: ChimeraX/ipSAE npz + heatmap { display-mode: "form" }
# For each (job, seed) picks the top-ranked sample and reads its PAE straight from the
# full-data JSON (key 'token_pair_pae', written when --need_atom_confidence true) - OpenDDE
# needs no PAE-capture patch. Writes into that seed's pae/ folder:
#   pae_<cifstem>.npz        (key 'pae', per-token; ligand-safe for ChimeraX>=1.10 + ipSAE)
#   <job>_<seed>_pae.json    (ColabFold schema)     <job>_<seed>_pae.png (heatmap w/ chain lines)
# Models are located via the summary JSONs (CIF derived from them), so naming/layout don't matter.
import os, glob, json, re
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
try:
    import gemmi; _HAS_GEMMI = True
except Exception:
    _HAS_GEMMI = False

_RUN = globals().get('RUN_DIR', OUT_DIR)
round_dp = 2  #@param {type:"integer"}

def _seed_of(p):
    for x in p.split(os.sep):
        if x.startswith('seed_'):
            return x.split('seed_')[-1]
    for x in p.split(os.sep):
        if x.isdigit():
            return x
    return 'NA'

def _cif_for(jf):
    cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
    if os.path.exists(cif):
        return cif
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    d = os.path.dirname(jf)
    cand = (glob.glob(os.path.join(d, f'*sample_{ms.group(1)}.cif')) if ms
            else glob.glob(os.path.join(d, '*.cif')))
    return cand[0] if cand else None

def _fulldata_for(cif):
    fd = cif.replace('_sample_', '_full_data_sample_').replace('.cif', '.json')
    if os.path.exists(fd):
        return fd
    ms = re.search(r'sample_(\d+)', os.path.basename(cif))
    cand = (glob.glob(os.path.join(os.path.dirname(cif), f'*full_data*sample_{ms.group(1)}*.json'))
            if ms else [])
    return cand[0] if cand else None

def _load_pae(fd):
    try:
        d = json.load(open(fd))
    except Exception:
        return None
    for k in ('token_pair_pae', 'pae', 'predicted_aligned_error'):
        if k in d:
            return np.array(d[k], dtype='float32')
    return None

def _seed_dir_of(cif):
    d = os.path.dirname(cif)
    return os.path.dirname(d) if os.path.basename(d) == 'predictions' else d

def _chain_boundaries(cif):
    if not (_HAS_GEMMI and cif and os.path.exists(cif)):
        return []
    try:
        st = gemmi.read_structure(cif); m = st[0]
        bnd, c = [], 0
        for ch in m:
            c += sum(1 for _ in ch); bnd.append(c)
        return bnd[:-1]
    except Exception:
        return []

def _heatmap(arr, cif, title, path):
    bnd = _chain_boundaries(cif)
    plt.figure(figsize=(5.2, 4.4))
    im = plt.imshow(arr, cmap='Greens_r', vmin=0, vmax=32)
    for b in bnd:
        plt.axhline(b - 0.5, color='k', lw=0.6); plt.axvline(b - 0.5, color='k', lw=0.6)
    plt.colorbar(im, label='PAE (Angstrom)')
    plt.title(title); plt.xlabel('Scored token'); plt.ylabel('Aligned token')
    plt.tight_layout(); plt.savefig(path, dpi=130); plt.close()

best = {}
for jf in glob.glob(os.path.join(_RUN, '**', '*summary_confidence*sample_*.json'), recursive=True):
    job = os.path.relpath(jf, _RUN).split(os.sep)[0]
    seed = _seed_of(jf)
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    if not ms:
        continue
    try:
        rank = float(json.load(open(jf)).get('ranking_score') or 0)
    except Exception:
        rank = 0.0
    k = (job, seed)
    if k not in best or rank > best[k]['rank']:
        best[k] = {'rank': rank, 'jf': jf, 'N': ms.group(1)}

consolidated = os.path.join(_RUN, '_pae'); os.makedirs(consolidated, exist_ok=True)
n_ok = n_missing = 0
for (job, seed), r in sorted(best.items()):
    cif = _cif_for(r['jf'])
    if not cif:
        continue
    fd  = _fulldata_for(cif)
    arr = _load_pae(fd) if fd else None
    if arr is None:
        n_missing += 1
        continue
    pae_dir = os.path.join(_seed_dir_of(cif), 'pae'); os.makedirs(pae_dir, exist_ok=True)
    cifstem = os.path.basename(cif)[:-4]
    np.savez_compressed(os.path.join(pae_dir, f'pae_{cifstem}.npz'), pae=arr.astype('float32'))
    json.dump([{'predicted_aligned_error': np.round(arr, round_dp).tolist(),
                'max_predicted_aligned_error': float(arr.max())}],
              open(os.path.join(pae_dir, f'{job}_{seed}_pae.json'), 'w'))
    _heatmap(arr, cif, f'{job}  seed {seed}', os.path.join(pae_dir, f'{job}_{seed}_pae.png'))
    np.savez_compressed(os.path.join(consolidated, f'{job}_seed{seed}_pae.npz'),
                        pae=arr.astype('float16'))
    n_ok += 1

print(f'Best-per-seed: {n_ok} ChimeraX/ipSAE npz + JSON + heatmaps, each in <job>/seed_<seed>/pae/.')
print('Consolidated npz copies ->', consolidated)
if n_missing:
    print(f'[!] {n_missing} model(s) had no full-data JSON - set need_atom_confidence=True in '
          'Run settings and rerun the inference cell.')

if best and n_ok:
    (job, seed), r = max(best.items(), key=lambda kv: kv[1]['rank'])
    png = glob.glob(os.path.join(_RUN, job, '**', f'{job}_{seed}_pae.png'), recursive=True)
    if png:
        from IPython.display import Image, display
        print(f'Top model: {job} seed {seed} (rank {r["rank"]:.3f})')
        display(Image(filename=png[0]))
print('ChimeraX >=1.10 (ligand-safe):  open <cif>  then  open pae_<cifstem>.npz structure #1')
print('ChimeraX older / protein-only:  open <cif>  then  open <job>_<seed>_pae.json format pae')

In [ ]:
#@title B6. ipSAE interface scores (Dunbrack, Boltz mode) { display-mode: "form" }
# Runs Dunbrack's ipsae.py on each best-per-seed model using the Boltz-style npz + cif
# (per-token, ligand-aware). Collects chain-pair ipSAE / ipTM / pDockQ / LIS into a CSV.
import os, glob, json, shutil, subprocess, sys
import numpy as np, pandas as pd

pae_cutoff  = 10   #@param {type:"number"}
dist_cutoff = 10   #@param {type:"number"}
_RUN = globals().get('RUN_DIR', OUT_DIR)

# --- fetch ipsae.py once (Colab can reach raw.githubusercontent.com) ---
IPSAE = '/content/ipsae.py'
if not (os.path.exists(IPSAE) and os.path.getsize(IPSAE) > 5000):
    for url in ('https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py',
                'https://raw.githubusercontent.com/DunbrackLab/IPSAE/master/ipsae.py'):
        os.system(f'wget -q -O {IPSAE} "{url}"')
        if os.path.exists(IPSAE) and os.path.getsize(IPSAE) > 5000:
            break
assert os.path.exists(IPSAE) and os.path.getsize(IPSAE) > 5000, 'Could not download ipsae.py'
print('ipsae.py ready:', os.path.getsize(IPSAE), 'bytes')

def _seed_of(path):
    for p in path.split(os.sep):
        if p.startswith('seed_'):
            return p.split('seed_')[-1]
    for p in path.split(os.sep):
        if p.isdigit():
            return p
    return 'NA'

ps = f'{int(pae_cutoff):02d}'
ds = f'{int(dist_cutoff):02d}'
rows, n_run, n_fail = [], 0, 0

for npz in sorted(glob.glob(os.path.join(_RUN, '**', 'pae', 'pae_*.npz'), recursive=True)):
    if os.path.basename(npz).endswith('_pae_best.npz'):
        continue
    cifstem = os.path.basename(npz)[4:-4]                 # strip 'pae_' and '.npz'
    job = os.path.relpath(npz, _RUN).split(os.sep)[0]
    seed = _seed_of(npz)
    seed_dir = os.path.dirname(os.path.dirname(npz))      # .../seed_<seed>  (parent of pae/)
    cifs = glob.glob(os.path.join(seed_dir, '**', cifstem + '.cif'), recursive=True)
    if not cifs:
        continue
    cif = cifs[0]; cifdir = os.path.dirname(cif); pae_dir = os.path.dirname(npz)

    r = subprocess.run([sys.executable, IPSAE, npz, cif, str(pae_cutoff), str(dist_cutoff)],
                       capture_output=True, text=True)
    n_run += 1
    out_txt = os.path.join(cifdir, f'{cifstem}_{ps}_{ds}.txt')
    if not os.path.exists(out_txt):
        alt = glob.glob(os.path.join(cifdir, f'{cifstem}_*_*.txt'))
        alt = [a for a in alt if not a.endswith('_byres.txt')]
        out_txt = alt[0] if alt else None
    if not out_txt or not os.path.exists(out_txt):
        n_fail += 1
        if n_fail <= 3:
            print(f'  no ipSAE output: {cifstem}\n   {r.stderr.strip()[-300:]}')
        continue

    # tidy: move ipsae outputs (.txt/_byres.txt/.pml) into the pae/ folder
    for f in glob.glob(os.path.join(cifdir, f'{cifstem}_{ps}_{ds}*')):
        try:
            shutil.move(f, os.path.join(pae_dir, os.path.basename(f)))
        except Exception:
            pass
    txt = os.path.join(pae_dir, os.path.basename(out_txt))
    if not os.path.exists(txt):
        txt = out_txt

    with open(txt) as fh:
        lines = [l.rstrip() for l in fh if l.strip()]
    hdr = next((i for i, l in enumerate(lines) if 'ipSAE' in l and ('Chn1' in l or l.split()[0] == 'Chn1')), None)
    if hdr is None:
        continue
    cols = lines[hdr].split()
    for l in lines[hdr + 1:]:
        parts = l.split()
        if len(parts) != len(cols):
            continue
        d = dict(zip(cols, parts)); d['job'] = job; d['seed'] = seed
        rows.append(d)

print(f'ipSAE run on {n_run} models ({n_fail} without output).')

if rows:
    df = pd.DataFrame(rows)
    keep_str = {'job', 'seed', 'Chn1', 'Chn2', 'Type', 'Model'}
    for c in df.columns:
        if c not in keep_str:
            conv = pd.to_numeric(df[c], errors='coerce')
            if conv.notna().any():
                df[c] = conv
    lead = [c for c in ['job', 'seed', 'Chn1', 'Chn2'] if c in df.columns]
    df = df[lead + [c for c in df.columns if c not in lead]]
    out_csv = os.path.join(_RUN, 'ipsae_all_chain_pairs.csv')
    df.to_csv(out_csv, index=False)
    print('Wrote', out_csv, '| rows:', len(df))

    if 'ipSAE' in df.columns:
        best = (df.sort_values('ipSAE', ascending=False)
                  .groupby('job', as_index=False).head(1)
                  .sort_values('ipSAE', ascending=False))
        best.to_csv(os.path.join(_RUN, 'ipsae_best_per_job.csv'), index=False)
        show = [c for c in ['job', 'seed', 'Chn1', 'Chn2', 'ipSAE', 'ipSAE_d0chn',
                            'ipTM_d0chn', 'pDockQ', 'LIS'] if c in best.columns]
        from IPython.display import display
        display(best[show].head(40))
else:
    print('No ipSAE rows parsed - check a sample .txt in a pae/ folder and the stderr above.')

In [ ]:
#@title 10. Zip & download all outputs { display-mode: "form" }
import shutil, os
job = JOB_SPEC['name']
zip_base = os.path.join(WORK, job + '_opendde')
_RUN = globals().get('RUN_DIR', OUT_DIR)
shutil.make_archive(zip_base, 'zip', os.path.join(_RUN, job))
zp = zip_base + '.zip'
print('Zipped:', zp, f'({os.path.getsize(zp)/1e6:.1f} MB)')
try:
    from google.colab import files
    files.download(zp)
except Exception as e:
    print('Download manually from the Files panel:', zp, '|', e)

In [ ]:
#@title 10b. Archive this run (date prefix + seed suffix, no overwrites) { display-mode: "form" }
# Protenix overwrites OUT_DIR/<job>/seed_*/predictions/<job>_sample_N.* on every rerun, so reruns
# clobber earlier results. This copies the current outputs into a timestamped archive folder with
# self-describing names: <STAMP>_<job>_sample<N>_seed<seed>.<ext>  (date prefix, seed suffix).
# Originals are left untouched. Works after a single run (JOB_SPEC) or a batch (BATCH_NAMES).
import os, glob, shutil, datetime, zipfile

make_zip = True  #@param {type:"boolean"}

STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M')

if 'BATCH_NAMES' in globals() and BATCH_NAMES:
    jobs = list(BATCH_NAMES)
elif 'JOB_SPEC' in globals() and JOB_SPEC:
    jobs = [JOB_SPEC['name']]
else:
    raise SystemExit('Nothing to archive: run a batch (B1/B2) or set JOB_SPEC first.')

_RUN = globals().get('RUN_DIR', OUT_DIR)
archive_root = os.path.join(OUT_DIR, '_archive', STAMP)
n = 0
for nm in jobs:
    jdir = os.path.join(_RUN, nm)
    if not os.path.isdir(jdir):
        continue
    for f in glob.glob(os.path.join(jdir, '**', '*'), recursive=True):
        if not os.path.isfile(f):
            continue
        seed = next((p.split('seed_')[-1] for p in f.split(os.sep) if p.startswith('seed_')), 'NA')
        stem, ext = os.path.splitext(os.path.basename(f))
        dst_dir = os.path.join(archive_root, nm)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy2(f, os.path.join(dst_dir, f'{STAMP}_{stem}_seed{seed}{ext}'))
        n += 1

print(f'Archived {n} files from {len(jobs)} job(s) -> {archive_root}')

if make_zip:
    zpath = archive_root + '.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(os.path.join(archive_root, '**', '*'), recursive=True):
            if os.path.isfile(f):
                z.write(f, os.path.relpath(f, archive_root))
    print(f'Zipped -> {zpath} ({os.path.getsize(zpath)/1e6:.1f} MB)')
    try:
        from google.colab import files
        files.download(zpath)
    except Exception as e:
        print('Download from the Files panel:', zpath, '|', e)